Training + TFLite Export

In [ ]:
# --- STEP 0: MEMORY & SPEED OPTIMIZATION ---
import tensorflow as tf
from tensorflow.keras import mixed_precision
import numpy as np
import os

# Enable mixed precision for T4 GPU efficiency
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

In [ ]:
# --- STEP 1: INITIALIZATION & DRIVE ---
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# --- STEP 2: DATA EXTRACTION ---
# Unzip to local storage (faster than reading from Drive)
if not os.path.exists('/content/dataset/'):
    !unzip -q '/content/drive/MyDrive/Colab Notebooks/Agri-tech/dataset.zip' -d /content/dataset/

In [ ]:
# --- STEP 3: PATHS & PARAMETERS ---
train_dir = '/content/dataset/New Plant Diseases Dataset(Augmented)/train'
valid_dir = '/content/dataset/New Plant Diseases Dataset(Augmented)/valid'

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

In [ ]:
# --- STEP 4: DATA PIPELINE (RAM SAFE) ---
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    valid_dir,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    label_mode='categorical'
)

# Get class names for the labels file later
class_names = train_ds.class_names

# Optimize pipeline
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

Found 140590 files belonging to 38 classes.
Found 35144 files belonging to 38 classes.


In [ ]:
# --- STEP 5: MODEL BUILDING (MobileNetV2) ---
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224, 224, 3)),
    tf.keras.layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.2),
    # Final layer must be float32 for TFLite compatibility
    tf.keras.layers.Dense(len(class_names), activation='softmax', dtype='float32')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
# --- STEP 6: TRAINING ---
model.fit(
    train_ds,
    epochs=10,
    validation_data=val_ds,
    validation_steps=20 # Limits RAM usage during validation
)

Epoch 1/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 274s 54ms/step - accuracy: 0.9184 - loss: 0.2813 - val_accuracy: 0.9594 - val_loss: 0.1178
Epoch 2/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 212s 36ms/step - accuracy: 0.9560 - loss: 0.1322 - val_accuracy: 0.9641 - val_loss: 0.1143
Epoch 3/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 155s 35ms/step - accuracy: 0.9617 - loss: 0.1113 - val_accuracy: 0.9672 - val_loss: 0.1008
Epoch 4/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 156s 35ms/step - accuracy: 0.9660 - loss: 0.0994 - val_accuracy: 0.9625 - val_loss: 0.1123
Epoch 5/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 156s 36ms/step - accuracy: 0.9673 - loss: 0.0955 - val_accuracy: 0.9578 - val_loss: 0.1059
Epoch 6/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 200s 35ms/step - accuracy: 0.9690 - loss: 0.0901 - val_accuracy: 0.9672 - val_loss: 0.1184
Epoch 7/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 201s 35ms/step - accuracy: 0.9699 - loss: 0.0870 - val_accuracy: 0.9609 - val_loss: 0.1244
Epoch 8/10
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 202s 46ms/step - accuracy: 

In [ ]:
# --- STEP 7: CONVERT TO TFLITE (FLOAT16 QUANTIZATION) ---
print("\nConverting model to TFLite...")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16] # Quantize to 16-bit floats
# Add this line to allow select TensorFlow ops (Flex ops)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]

tflite_model = converter.convert()

# Save the TFLite model
with open('model.tflite', 'wb') as f:
    f.write(tflite_model)


Converting model to TFLite...
Saved artifact at '/tmp/tmphmhek8pv'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_155')
Output Type:
  TensorSpec(shape=(None, 38), dtype=tf.float32, name=None)
Captures:
  136891823502160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136891823502544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136891823504848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136891823504464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136891823503888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136891823502352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136891823504272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136891823504080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136891823503696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136891823502736: TensorSpec(shape=(), dtype=tf.r

In [ ]:
# --- STEP 8: CREATE LABELS.TXT ---
# Flutter apps need this file to map index numbers to plant names
with open('labels.txt', 'w') as f:
    for name in class_names:
        f.write(name + '\n')

print("\nSuccess! Download 'model.tflite' and 'labels.txt' from the files tab.")


Success! Download 'model.tflite' and 'labels.txt' from the files tab.


In [ ]:
import tensorflow as tf
import numpy as np
import os

# --- STEP 9: EVALUATE TFLITE MODEL ACCURACY ---
print("\nEvaluating TFLite model...")

# Re-define necessary variables from previous steps to ensure they are available
train_dir = '/content/dataset/New Plant Diseases Dataset(Augmented)/train'
valid_dir = '/content/dataset/New Plant Diseases Dataset(Augmented)/valid'

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Re-create val_ds for evaluation
val_ds = tf.keras.utils.image_dataset_from_directory(
    valid_dir,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    label_mode='categorical'
)

AUTOTUNE = tf.data.AUTOTUNE
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

model_path='/content'
# Load the TFLite model and allocate tensors
interpreter = tf.lite.Interpreter(model_path='model.tflite')
interpreter.allocate_tensors()

# Get input and output tensors details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Lists to store true labels and predictions
true_labels = []
predictions = []

# Iterate over the validation dataset
for images, labels in val_ds:
    # Iterate over each image and label in the batch
    for i in range(images.shape[0]):
        # Prepare input tensor for a single image
        # TFLite models expect float32 inputs by default, if not specified otherwise
        # Use [i:i+1] to keep the batch dimension, even if it's 1
        input_tensor = tf.cast(images[i:i+1], tf.float32)
        interpreter.set_tensor(input_details[0]['index'], input_tensor)

        # Run inference
        interpreter.invoke()

        # Get output tensor (output will also have a batch dimension of 1)
        output = interpreter.get_tensor(output_details[0]['index'])

        # Store true label and prediction for this single image
        true_labels.append(np.argmax(labels.numpy()[i]))
        predictions.append(np.argmax(output[0]))

# Convert lists to numpy arrays for accuracy calculation
true_labels = np.array(true_labels)
predictions = np.array(predictions)

# Calculate accuracy
accuracy = np.mean(true_labels == predictions)
print(f"TFLite Model Accuracy on Validation Set: {accuracy * 100:.2f}%")


Evaluating TFLite model...
Found 35144 files belonging to 38 classes.


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:
import tflite_support

print("Attributes and submodules of tflite_support:")
for attr_name in dir(tflite_support):
    if not attr_name.startswith('_'): # Exclude private attributes
        attr = getattr(tflite_support, attr_name)
        if isinstance(attr, type(tflite_support)): # Check if it's a module
            print(f"- Module: {attr_name}")
        else:
            print(f"- Attribute: {attr_name}")


Attributes and submodules of tflite_support:


In [ ]:
from tflite_support.metadata_writers import image_classifier
from tflite_support.metadata_writers import writer_utils

# --- CONFIGURATION PATHS ---
model_path = "/content/model.tflite"          # The raw model you uploaded
label_path = "/content/labels.txt"            # The text file with your crop diseases
save_path = "model_with_metadata.tflite"  # The brand new output model

# In our dart code, we were using 127.5 for Mean and Standard Deviation.
# This tells ML Kit structurally how to normalize the image RGB pixels on the phone!
_INPUT_NORM_MEAN = 127.5
_INPUT_NORM_STD = 127.5

print("Starting Metadata Injection...")

try:
    # 1. Create the specialized Image Classification Writer
    writer = image_classifier.MetadataWriter.create_for_inference(
        writer_utils.load_file(model_path),
        [_INPUT_NORM_MEAN],
        [_INPUT_NORM_STD],
        [label_path]
    )

    # 2. Synthesize and write the new TFLite binary file
    writer_utils.save_file(writer.populate(), save_path)

    print("\n✅ SUCCESS: Metadata injected natively!")
    print("\nBelow is the injected Metadata JSON verifying ML Kit compatibility:")
    print(writer.get_metadata_json())

except Exception as e:
    print(f"\n❌ FAILED TO INJECT METADATA: {e}")

ModuleNotFoundError: No module named 'tflite_support.metadata_writers'